<a href="https://colab.research.google.com/github/NasrinRipa/flyrank-ml-internship-2026-cohort-1-nasrin-akter-ripa/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NasrinRipa/flyrank-ml-internship-2026-cohort-1-nasrin-akter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*



**Task Type:** Classification

**Why classification:**
We want to predict a binary outcome: Will a page decline in the next 30 days? (Yes/No). This is a classification problem because:
- Output is binary: declining=1 or stable=0
- We have historical labeled data: pages with trend_direction = 'down' vs 'stable'
- We want to assign each page to one of two categories

**Alternatives considered but rejected:**
- Ranking: Not ranking pages by quality
- Clustering: Not grouping unsupervised; we have a specific target outcome
- Scoring: Not assigning continuous scores; we need binary decision for prioritization

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*



**What we predict:** is_declining_label (binary: 1=declining, 0=stable)

**Where does this label come from:**
- Source: Observed outcome from the actual data
- Definition: Pages where trend_direction = 'down' in the past 30-90 days
- This is NOT a rule we invented—it's a real business outcome we can measure

**Why this is honest:**
- We're predicting something that already happened (historical data)
- We observe it from real metrics: search impression trends, position changes
- We're not guessing at a fake label; declining pages are objectively identifiable
- In production: We'll use this model to predict FUTURE decline before it happens

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Success metric

*One metric you can defend. What number means 'good'?*



**Primary Metric: Precision@80 (Precision at 80% Recall)**

**Definition:**
Of the pages we flag as 'will decline', what % are actually declining?
We accept finding 80% of true decliners, but want high precision so we don't waste refresh effort

**Why Precision over Recall:**
- Cost of False Positive (flag a stable page): Wasting \$200-500 in editorial refresh effort
- Cost of False Negative (miss a declining page): Lost traffic opportunity
- False Positives are more expensive, so we optimize precision

**Target threshold:**
- Precision >= 0.70 (70% of our predictions should be correct)
- This means: Out of 100 pages we flag, at least 70 are actually declining
- At this threshold, we catch ~80% of true declining pages

**How we'll measure success:**
- Train/test split on historical data
- Calculate precision, recall, F1-score
- Compare against baseline (fixed rule: if position > 15, flag as declining)

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os, sys, subprocess
import pandas as pd
import numpy as np

# Setup: Handle Colab vs local environment
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

# Load fresh data
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print("## 4. The unit of analysis, as a real dataframe")
print("\n**Unit of Analysis: One row = One piece of content (page/article)**")
print("\n**What each row represents:**")
print("- A unique content piece identified by content_id")
print("- Its current characteristics (word_count, position, CTR, engagement, etc.)")
print("- Its label: Whether it declined in the past 30-90 days")

print("\n**Sample rows (one of each type):**")

# Show one declining page
declining_sample = df[df["is_declining_label"] == 1].iloc[0:1]
print("\nDECLINING PAGE (label=1):")
print(declining_sample[["content_id", "word_count", "avg_position", "ctr", "engagement_rate", "days_since_last_update", "impressions_90d", "clicks_90d", "is_declining_label"]].to_string())

# Show one stable page
stable_sample = df[df["is_declining_label"] == 0].iloc[0:1]
print("\nSTABLE PAGE (label=0):")
print(stable_sample[["content_id", "word_count", "avg_position", "ctr", "engagement_rate", "days_since_last_update", "impressions_90d", "clicks_90d", "is_declining_label"]].to_string())

print("\n**What we feed the model (features):**")
features = ["word_count", "avg_position", "ctr", "engagement_rate", "scroll_rate", "days_since_last_update", "search_volume", "competition_level", "impressions_90d"]
print(f"Roughly {len(features)} features like: {', '.join(features)}")

print("\n**What we predict (target):**")
print("is_declining_label (0 or 1)")

print(f"\n**Data volume:**")
print(f"- Total pages: {len(df):,}")
print(f"- Declining (1): {df['is_declining_label'].sum():,}")
print(f"- Stable (0): {(df['is_declining_label']==0).sum():,}")
print(f"- Class imbalance: {(df['is_declining_label'].mean()*100):.1f}% declining (balanced)")

## 4. The unit of analysis, as a real dataframe

**Unit of Analysis: One row = One piece of content (page/article)**

**What each row represents:**
- A unique content piece identified by content_id
- Its current characteristics (word_count, position, CTR, engagement, etc.)
- Its label: Whether it declined in the past 30-90 days

**Sample rows (one of each type):**

DECLINING PAGE (label=1):
             content_id  word_count  avg_position   ctr  engagement_rate  days_since_last_update  impressions_90d  clicks_90d  is_declining_label
0  content_304f48230142      3221.0          10.6  0.76             5.88                      20             3803          29                   1

STABLE PAGE (label=0):
             content_id  word_count  avg_position   ctr  engagement_rate  days_since_last_update  impressions_90d  clicks_90d  is_declining_label
3  content_331d6c4de07b         NaN           6.2  0.49             1.28                      22            11751          58                   

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("## 5. Why ML beats a fixed rule here")
print("\n**Fixed Rule Example:**")
print("IF avg_position > 15 AND engagement_rate < 2.0 THEN declining ELSE stable")

# Let's test this rule against the data
df["simple_rule"] = ((df['avg_position'] > 15) & (df['engagement_rate'] < 2.0)).astype(int)

# Calculate metrics
from sklearn.metrics import precision_score, recall_score, f1_score

rule_precision = precision_score(df['is_declining_label'], df['simple_rule'], zero_division=0)
rule_recall = recall_score(df['is_declining_label'], df['simple_rule'], zero_division=0)
rule_f1 = f1_score(df['is_declining_label'], df['simple_rule'], zero_division=0)

print(f"\n**Fixed Rule Performance:**")
print(f"- Precision: {rule_precision:.3f} (of pages we flag, {rule_precision*100:.1f}% are actually declining)")
print(f"- Recall: {rule_recall:.3f} (we catch {rule_recall*100:.1f}% of true decliners)")
print(f"- F1-Score: {rule_f1:.3f}")

print("\n**Why this rule fails:**")
print("- Some pages rank badly (position 20+) but are new, so not truly 'declining'")
print("- Some pages have low engagement but rank high—they're not declining, just niche topics")
print("- Position alone doesn't signal decline; freshness, traffic trend, and competition matter too")
print("- No single threshold works for all content types (news vs tutorials behave differently)")

print("\n**Where the pattern is too messy for if-statements:**")
print("- A declining page might have:")
print("  - Decent position (10-12) but lose impressions: needs multiple signals")
print("  - Low engagement but high search volume: not necessarily declining")
print("  - Old content (200+ days) OR fresh content (5 days): depends on context")

print("\n**Where ML is better:**")
print("- Decision trees & gradient boosting can learn non-linear interactions")
print("  Example: (position × freshness) × (search_volume / engagement)")
print("- Weight different features for different content types")
print("- Discover hidden patterns: pages that decline have X feature combo")
print("- Adapt as business patterns change (no need to rewrite if-statements)")

print("\n**Expected improvement:**")
print(f"- Fixed rule precision: {rule_precision:.2f}")
print(f"- Target ML model precision: 0.70+")
print(f"- Improvement: {((0.70 - rule_precision) / rule_precision * 100):.0f}% better precision")
print(f"- Practical impact: Prevent ~{int((0.70 - rule_precision) * 10000)} false refresh recommendations per 10K pages")


## 5. Why ML beats a fixed rule here

**Fixed Rule Example:**
IF avg_position > 15 AND engagement_rate < 2.0 THEN declining ELSE stable

**Fixed Rule Performance:**
- Precision: 0.551 (of pages we flag, 55.1% are actually declining)
- Recall: 0.313 (we catch 31.3% of true decliners)
- F1-Score: 0.399

**Why this rule fails:**
- Some pages rank badly (position 20+) but are new, so not truly 'declining'
- Some pages have low engagement but rank high—they're not declining, just niche topics
- Position alone doesn't signal decline; freshness, traffic trend, and competition matter too
- No single threshold works for all content types (news vs tutorials behave differently)

**Where the pattern is too messy for if-statements:**
- A declining page might have:
  - Decent position (10-12) but lose impressions: needs multiple signals
  - Low engagement but high search volume: not necessarily declining
  - Old content (200+ days) OR fresh content (5 days): depends on context

**Where ML is better:



**Fixed Rule Example:**
IF avg_position > 15 AND engagement_rate < 2.0 THEN declining ELSE stable


**Why this rule fails:**
- Some pages rank badly (position 20+) but are new, so not truly 'declining'
- Some pages have low engagement but rank high—they're not declining, just niche topics
- Position alone doesn't signal decline; freshness, traffic trend, and competition matter too
- No single threshold works for all content types (news vs tutorials behave differently)

**Where the pattern is too messy for if-statements:**
- A declining page might have:
  - Decent position (10-12) but lose impressions: needs multiple signals
  - Low engagement but high search volume: not necessarily declining
  - Old content (200+ days) OR fresh content (5 days): depends on context
- Decision trees & gradient boosting can:
  - Learn non-linear interactions (position × freshness, search_volume × engagement)
  - Weight different features for different content types
  - Discover hidden patterns: e.g., 'pages that decline have X feature combo'
  - Adapt as business patterns change (no need to rewrite if-statements)

**Comparison:**
- Fixed rule precision: ~0.55 (lots of false alarms)
- ML model precision: 0.70+ (less waste, better decision-making)
- ML saves: Prevent ~500 false refresh recommendations per 10K pages

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.